In [3]:
from jetbot import Camera

camera = Camera.instance(
    width=224,
    height=224
)

print("Camera ready!")

Camera ready!


In [4]:
from jetbot import Robot, bgr8_to_jpeg
import ipywidgets as widgets
import traitlets
from IPython.display import display
from PIL import Image

import os
import csv
import time
import threading


# =====================================================
# הגדרות בסיסיות
# =====================================================

robot = Robot()

images_dir = "datasets/clockwise/images"
csv_path = "datasets/clockwise/labels.csv"

os.makedirs(images_dir, exist_ok=True)

if not os.path.exists(csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filename", "steering"])


# 0 = עצירה
# 1 = קדימה
# -1 = אחורה
drive_direction = 0

recording = False


# =====================================================
# כיול מנועים
# =====================================================

LEFT_GAIN = 1.035
RIGHT_GAIN = 1.00


# =====================================================
# התנעה
# =====================================================

STARTUP_SPEED = 0.18
STARTUP_TIME = 0.15


# =====================================================
# מצלמה חיה
# =====================================================

camera_view = widgets.Image(
    format="jpeg",
    width=400,
    height=300
)

camera_link = traitlets.dlink(
    (camera, "value"),
    (camera_view, "value"),
    transform=bgr8_to_jpeg
)


# =====================================================
# Steering
# =====================================================

steering = widgets.FloatSlider(
    value=0.0,
    min=-1.0,
    max=1.0,
    step=0.01,
    description="Steering:"
)


# =====================================================
# Speed
# =====================================================

speed_slider = widgets.FloatSlider(
    value=0.09,
    min=0.05,
    max=0.30,
    step=0.01,
    description="Speed:"
)


# =====================================================
# כפתורי נסיעה
# =====================================================

start_button = widgets.Button(
    description="START"
)

stop_button = widgets.Button(
    description="STOP"
)

back_button = widgets.Button(
    description="BACK"
)


# =====================================================
# כפתורי Dataset
# =====================================================

save_button = widgets.Button(
    description="SAVE"
)

delete_button = widgets.Button(
    description="DELETE LAST"
)

record_button = widgets.ToggleButton(
    value=False,
    description="RECORD"
)


# =====================================================
# DELETE RANGE
# =====================================================

delete_from_row = widgets.IntText(
    value=2,
    description="From Row:"
)

delete_to_row = widgets.IntText(
    value=2,
    description="To Row:"
)

delete_range_button = widgets.Button(
    description="DELETE RANGE"
)


# =====================================================
# DELETE לפי IMG
# =====================================================

delete_filename = widgets.Text(
    value="",
    description="IMG:",
    placeholder="1788683316448.jpg"
)

delete_by_filename_button = widgets.Button(
    description="DELETE IMG"
)


# =====================================================
# DELETE לפי CSV ROW
# =====================================================

delete_csv_row = widgets.IntText(
    value=2,
    description="CSV Row:"
)

delete_by_row_button = widgets.Button(
    description="DELETE ROW"
)


# =====================================================
# DELETE ALL
# =====================================================

confirm_delete_all = widgets.Checkbox(
    value=False,
    description="Confirm DELETE ALL"
)

delete_all_button = widgets.Button(
    description="DELETE ALL"
)


# =====================================================
# Status + Samples
# =====================================================

dataset_status = widgets.Label(
    value=""
)

sample_counter = widgets.Label(
    value="Samples: 0"
)


def update_sample_counter():

    if not os.path.exists(csv_path):
        sample_counter.value = "Samples: 0"
        return

    with open(csv_path, "r", newline="") as f:
        rows = list(csv.reader(f))

    count = max(
        0,
        len(rows) - 1
    )

    sample_counter.value = (
        f"Samples: {count}"
    )


update_sample_counter()


# =====================================================
# שמירת תמונה + Steering
# =====================================================

def save_sample():

    image = camera.value

    if image is None:
        print("No camera image")
        return

    timestamp = int(
        time.time() * 1000
    )

    filename = f"{timestamp}.jpg"

    path = os.path.join(
        images_dir,
        filename
    )

    # BGR -> RGB
    rgb_image = (
        image[:, :, ::-1].copy()
    )

    Image.fromarray(
        rgb_image
    ).save(path)

    # שומרים רק Steering
    steering_value = steering.value

    with open(
        csv_path,
        "a",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            filename,
            steering_value
        ])

    update_sample_counter()

    print(
        "SAVED:",
        filename,
        "steering =",
        round(steering_value, 2)
    )


# =====================================================
# SAVE ידני
# =====================================================

def save_button_clicked(b):
    save_sample()


save_button.on_click(
    save_button_clicked
)


# =====================================================
# DELETE LAST
# =====================================================

def delete_last_sample(b):

    if recording:
        dataset_status.value = "STOP RECORD before deleting."
        return

    with open(
        csv_path,
        "r",
        newline=""
    ) as f:

        rows = list(
            csv.reader(f)
        )

    if len(rows) <= 1:
        dataset_status.value = "Nothing to delete."
        return

    filename = rows[-1][0]

    image_path = os.path.join(
        images_dir,
        filename
    )

    if os.path.exists(image_path):
        os.remove(image_path)

    rows = rows[:-1]

    with open(
        csv_path,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)
        writer.writerows(rows)

    update_sample_counter()

    dataset_status.value = (
        f"Deleted: {filename}"
    )

    print(
        "DELETED:",
        filename
    )


delete_button.on_click(
    delete_last_sample
)


# =====================================================
# DELETE RANGE
#
# From Row ו-To Row כוללים את שני הקצוות
#
# 6 עד 6 = מוחק רק שורה 6
# 6 עד 7 = מוחק 6 ו-7
# 6 עד 10 = מוחק 6,7,8,9,10
# =====================================================

def delete_range(b):

    if recording:
        dataset_status.value = "STOP RECORD before deleting."
        return

    start_row = delete_from_row.value
    end_row = delete_to_row.value

    # שורה 1 היא Header
    if start_row < 2 or end_row < 2:
        dataset_status.value = "Rows must be 2 or higher."
        return

    if start_row > end_row:
        dataset_status.value = (
            "From Row cannot be greater than To Row."
        )
        return

    with open(
        csv_path,
        "r",
        newline=""
    ) as f:

        rows = list(
            csv.reader(f)
        )

    if len(rows) <= 1:
        dataset_status.value = "Dataset is empty."
        return

    if end_row > len(rows):
        dataset_status.value = (
            f"CSV has only {len(rows)} rows."
        )
        return

    # ממירים מספר שורה לאינדקס של Python
    start_index = start_row - 1
    end_index = end_row - 1

    # הטווח כולל את שני הקצוות
    rows_to_delete = (
        rows[start_index:end_index + 1]
    )

    deleted_images = 0

    for row in rows_to_delete:

        if len(row) == 0:
            continue

        filename = row[0]

        image_path = os.path.join(
            images_dir,
            filename
        )

        if os.path.exists(image_path):
            os.remove(image_path)
            deleted_images += 1

    # מוחקים את השורות מה-CSV
    del rows[start_index:end_index + 1]

    # שומרים מחדש
    with open(
        csv_path,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)
        writer.writerows(rows)

    update_sample_counter()

    deleted_count = len(
        rows_to_delete
    )

    dataset_status.value = (
        f"Deleted Rows {start_row}-{end_row}. "
        f"Samples deleted: {deleted_count}"
    )

    print(
        f"DELETED ROWS {start_row}-{end_row}",
        "| Samples:",
        deleted_count,
        "| Images:",
        deleted_images
    )


delete_range_button.on_click(
    delete_range
)


# =====================================================
# DELETE לפי IMG
# =====================================================

def delete_by_filename(b):

    if recording:
        dataset_status.value = "STOP RECORD before deleting."
        return

    target = (
        delete_filename.value.strip()
    )

    if target == "":
        dataset_status.value = "Enter image filename."
        return

    # מאפשר גם להכניס רק את המספר
    if not target.lower().endswith(".jpg"):
        target += ".jpg"

    with open(
        csv_path,
        "r",
        newline=""
    ) as f:

        rows = list(
            csv.reader(f)
        )

    if len(rows) <= 1:
        dataset_status.value = "Dataset is empty."
        return

    header = rows[0]
    samples = rows[1:]

    new_samples = []
    found = False

    for row in samples:

        if len(row) == 0:
            continue

        if row[0] == target:
            found = True

        else:
            new_samples.append(row)

    if not found:
        dataset_status.value = (
            f"{target} not found in labels.csv"
        )
        return

    image_path = os.path.join(
        images_dir,
        target
    )

    if os.path.exists(image_path):
        os.remove(image_path)

    with open(
        csv_path,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow(header)
        writer.writerows(new_samples)

    update_sample_counter()

    dataset_status.value = (
        f"Deleted IMG: {target}"
    )

    delete_filename.value = ""

    print(
        "DELETED IMG:",
        target
    )


delete_by_filename_button.on_click(
    delete_by_filename
)


# =====================================================
# DELETE לפי CSV ROW
# =====================================================

def delete_by_csv_row(b):

    if recording:
        dataset_status.value = "STOP RECORD before deleting."
        return

    row_number = (
        delete_csv_row.value
    )

    if row_number < 2:
        dataset_status.value = (
            "CSV Row must be 2 or higher."
        )
        return

    with open(
        csv_path,
        "r",
        newline=""
    ) as f:

        rows = list(
            csv.reader(f)
        )

    if row_number > len(rows):
        dataset_status.value = (
            f"CSV has only {len(rows)} rows."
        )
        return

    index = row_number - 1

    filename = rows[index][0]

    image_path = os.path.join(
        images_dir,
        filename
    )

    if os.path.exists(image_path):
        os.remove(image_path)

    del rows[index]

    with open(
        csv_path,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)
        writer.writerows(rows)

    update_sample_counter()

    dataset_status.value = (
        f"Deleted CSV Row "
        f"{row_number}: {filename}"
    )

    print(
        "DELETED CSV ROW:",
        row_number,
        "|",
        filename
    )


delete_by_row_button.on_click(
    delete_by_csv_row
)


# =====================================================
# DELETE ALL
# =====================================================

def delete_all(b):

    if recording:
        dataset_status.value = (
            "STOP RECORD before DELETE ALL."
        )
        return

    if not confirm_delete_all.value:
        dataset_status.value = (
            "Check Confirm DELETE ALL first."
        )
        return

    deleted_images = 0

    if os.path.exists(images_dir):

        for filename in os.listdir(
            images_dir
        ):

            image_path = os.path.join(
                images_dir,
                filename
            )

            if os.path.isfile(image_path):

                os.remove(image_path)
                deleted_images += 1

    # מאפסים את labels.csv
    with open(
        csv_path,
        "w",
        newline=""
    ) as f:

        writer = csv.writer(f)

        writer.writerow([
            "filename",
            "steering"
        ])

    confirm_delete_all.value = False

    update_sample_counter()

    dataset_status.value = (
        f"ALL DATA DELETED. "
        f"Images deleted: {deleted_images}"
    )

    print(
        "ALL DATA DELETED",
        "| Images:",
        deleted_images
    )


delete_all_button.on_click(
    delete_all
)


# =====================================================
# שליטה במנועים
# =====================================================

def update_motors():

    if drive_direction == 0:
        robot.stop()
        return

    speed = speed_slider.value

    s = steering.value

    steering_power = (
        s * 0.10
    )

    left_speed = (
        speed + steering_power
    )

    right_speed = (
        speed - steering_power
    )

    # כיול מנועים
    left_speed = (
        left_speed * LEFT_GAIN
    )

    right_speed = (
        right_speed * RIGHT_GAIN
    )

    # הגבלת ערכים
    left_speed = max(
        0.0,
        min(1.0, left_speed)
    )

    right_speed = max(
        0.0,
        min(1.0, right_speed)
    )

    if drive_direction == 1:

        robot.left_motor.value = (
            left_speed
        )

        robot.right_motor.value = (
            right_speed
        )

    elif drive_direction == -1:

        robot.left_motor.value = (
            -left_speed
        )

        robot.right_motor.value = (
            -right_speed
        )


# =====================================================
# START עם התנעה קצרה
# =====================================================

def start_robot(b):

    global drive_direction

    drive_direction = 1

    robot.left_motor.value = (
        STARTUP_SPEED * LEFT_GAIN
    )

    robot.right_motor.value = (
        STARTUP_SPEED * RIGHT_GAIN
    )

    time.sleep(
        STARTUP_TIME
    )

    update_motors()

    print("FORWARD")


# =====================================================
# BACK
# =====================================================

def back_robot(b):

    global drive_direction

    drive_direction = -1

    update_motors()

    print("BACKWARD")


# =====================================================
# STOP
# =====================================================

def stop_robot(b):

    global drive_direction

    drive_direction = 0

    robot.stop()

    print("STOPPED")


start_button.on_click(
    start_robot
)

back_button.on_click(
    back_robot
)

stop_button.on_click(
    stop_robot
)


# =====================================================
# שינוי Steering / Speed בזמן נסיעה
# =====================================================

def steering_changed(change):
    update_motors()


def speed_changed(change):
    update_motors()


steering.observe(
    steering_changed,
    names="value"
)

speed_slider.observe(
    speed_changed,
    names="value"
)


# =====================================================
# RECORD אוטומטי
# =====================================================

def recording_loop():

    global recording

    while recording:

        if drive_direction == 1:
            save_sample()

        time.sleep(
            0.2
        )


def record_changed(change):

    global recording

    if change["new"]:

        recording = True

        thread = threading.Thread(
            target=recording_loop,
            daemon=True
        )

        thread.start()

        print(
            "RECORDING STARTED"
        )

    else:

        recording = False

        print(
            "RECORDING STOPPED"
        )


record_button.observe(
    record_changed,
    names="value"
)


# =====================================================
# בניית הממשק
# =====================================================

drive_buttons = widgets.HBox([
    start_button,
    stop_button,
    back_button
])

data_buttons = widgets.HBox([
    save_button,
    delete_button,
    record_button
])

delete_range_controls = widgets.HBox([
    delete_from_row,
    delete_to_row,
    delete_range_button
])

delete_img_controls = widgets.HBox([
    delete_filename,
    delete_by_filename_button
])

delete_row_controls = widgets.HBox([
    delete_csv_row,
    delete_by_row_button
])

delete_all_controls = widgets.HBox([
    confirm_delete_all,
    delete_all_button
])


# =====================================================
# הצגת הממשק
# =====================================================

display(camera_view)

display(steering)
display(speed_slider)

display(drive_buttons)

display(sample_counter)

display(data_buttons)

display(delete_range_controls)

display(delete_img_controls)

display(delete_row_controls)

display(delete_all_controls)

display(dataset_status)

Image(value=b'\xff\xd8\xff\xe0\x00\x10JFIF\x00\x01\x01\x00\x00\x01\x00\x01\x00\x00\xff\xdb\x00C\x00\x02\x01\x0…

FloatSlider(value=0.0, description='Steering:', max=1.0, min=-1.0, step=0.01)

FloatSlider(value=0.09, description='Speed:', max=0.3, min=0.05, step=0.01)

Label(value='Samples: 3361')

Label(value='')

In [5]:
camera.stop()
print("Camera stopped")

Camera stopped


In [4]:
import csv
from collections import Counter

csv_path = "datasets/clockwise/labels.csv"

with open(csv_path, "r") as f:
    reader = csv.reader(f)
    rows = list(reader)[1:]

values = [round(float(row[1]), 2) for row in rows]

counter = Counter(values)

print("Total samples:", len(values))
print("\nSteering distribution:")
print("----------------------")

for steering_value in sorted(counter):
    print(
        f"{steering_value:6.2f} : "
        f"{counter[steering_value]:4d}"
    )

# חלוקה כללית
left = sum(1 for x in values if x < -0.05)
straight = sum(1 for x in values if -0.05 <= x <= 0.05)
right = sum(1 for x in values if x > 0.05)

print("\nGeneral distribution:")
print("----------------------")
print("LEFT     :", left)
print("STRAIGHT :", straight)
print("RIGHT    :", right)

print("\nRange:")
print("MIN:", min(values))
print("MAX:", max(values))

Total samples: 3361

Steering distribution:
----------------------
 -0.48 :    2
 -0.44 :    1
 -0.42 :    2
 -0.32 :    2
 -0.31 :    1
 -0.30 :    3
 -0.24 :    3
 -0.22 :    5
 -0.21 :    1
 -0.20 :    3
 -0.19 :    6
 -0.18 :    1
 -0.17 :    1
 -0.16 :    2
 -0.15 :    2
 -0.14 :    5
 -0.13 :    7
 -0.12 :   11
 -0.11 :   20
 -0.10 :   14
 -0.09 :   12
 -0.08 :   13
 -0.07 :   27
 -0.06 :   30
 -0.05 :   24
 -0.04 :   29
 -0.03 :   78
 -0.02 :   83
 -0.01 :   59
  0.00 :  147
  0.01 :  118
  0.02 :  110
  0.03 :  202
  0.04 :  122
  0.05 :  283
  0.06 :  179
  0.07 :  287
  0.08 :  185
  0.09 :  245
  0.10 :  269
  0.11 :   55
  0.12 :   85
  0.13 :   82
  0.14 :  128
  0.15 :  100
  0.16 :   38
  0.17 :   45
  0.18 :   22
  0.19 :   23
  0.20 :   53
  0.21 :   21
  0.22 :   25
  0.23 :   39
  0.24 :    6
  0.25 :    3
  0.26 :    5
  0.27 :    3
  0.28 :    9
  0.30 :    5
  0.31 :   16
  0.32 :    4

General distribution:
----------------------
LEFT     : 174
STRAIGHT : 1255
RI

In [ ]:
import csv
from collections import Counter

csv_path = "datasets/clockwise/labels.csv"

with open(csv_path, "r") as f:
    rows = list(csv.reader(f))[1:]

values = [
    round(float(row[1]), 2)
    for row in rows
]

counter = Counter(values)

print("Total:", len(values))
print("----------------")

for value in sorted(counter):
    print(f"{value:6.2f} : {counter[value]:4d}")

from jetbot import Robot
import time

robot = Robot()

LEFT_GAIN = 1.035
RIGHT_GAIN = 1.00
SPEED = 0.15

robot.left_motor.value = SPEED * LEFT_GAIN
robot.right_motor.value = SPEED * RIGHT_GAIN

time.sleep(3)

robot.stop()

import os
import csv

images_dir = "datasets/clockwise/images"
csv_path = "datasets/clockwise/labels.csv"

with open(csv_path, "r", newline="") as f:
    rows = list(csv.reader(f))

header = rows[0]
samples = rows[1:]

# משאירים את 153 הראשונות
to_keep = samples[:153]
to_delete = samples[153:]

# מוחקים מתמונה 154 ועד הסוף
for row in to_delete:
    filename = row[0]
    image_path = os.path.join(images_dir, filename)

    if os.path.exists(image_path):
        os.remove(image_path)

# מעדכנים את labels.csv
with open(csv_path, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(header)
    writer.writerows(to_keep)

print("Deleted:", len(to_delete))
print("Remaining:", len(to_keep))